## Coordinated Behavior Detection

In [ ]:
## Real fraud is rarely solo. A single person buying fake likes is a nuisance. 
# A bot farm with 500 accounts hitting 85 videos simultaneously is organized crime. 
# We build three signals to detect coordination, then combine them into a single score.

## Signal 1: Peak Hour Clustering
#Bot farms often run on schedules -- same timezone, same shift, or same automated trigger time. We find hours where an
#unusually high number of suspicious videos peaked.

## Signal 2: Temporal Co-Spiking
# If 10 unrelated videos all had their maximum like spike within the same hour window, that's not coincidence. 
# That's a coordinated attack.

## Signal 3: Bot Network Clustering (DBSCAN)
# We take ONLY the 500 suspicious videos and cluster them by engagement style. Videos in the same cluster share the same fake
#engagement fingerprint -- suggesting they're from the same bot network using identical settings.

In [1]:
import pandas as pd
from sklearn.cluster import DBSCAN
from collections import Counter
import seaborn as sns

In [ ]:
# ============================================================
# 1. PEAK HOUR CLUSTERING — Bot farms may often spike on schedule
# ============================================================

In [8]:
yt_sample = pd.read_csv("data/processed/yt_sample.csv")

In [4]:
video_features = pd.read_csv("data/processed/engineered_video_features2.csv")

In [6]:
def detect_peak_hour_clusters(df, min_cluster_size=5):
    peak_counts = df['peak_hour'].value_counts()
    # Hours with unusually high activity (>1.5 std above mean)
    mean_peaks = peak_counts.mean()
    std_peaks = peak_counts.std()
    suspicious_hours = peak_counts[peak_counts > mean_peaks + 1.5 * std_peaks].index.tolist()
    df['is_suspicious_hour'] = df['peak_hour'].isin(suspicious_hours)
    hour_cluster_size = df.groupby('peak_hour').size().to_dict()
    df['peak_hour_cluster_size'] = df['peak_hour'].map(hour_cluster_size)
    return df, suspicious_hours
video_features, suspicious_hours = detect_peak_hour_clusters(video_features)
print(f"Suspicious peak hours (possible bot schedule): {sorted(suspicious_hours)}")
print(video_features['peak_hour'].value_counts().head(10))

Suspicious peak hours (possible bot schedule): [12]
peak_hour
12    889
21    752
20    688
22    675
11    616
19    582
0     524
16    520
23    502
17    495
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 7: TEMPORAL CO-SPIKING
# ============================================================

In [9]:
def detect_cospiking(df_raw, window_hours=2):
    results = []
    for video_id, group in df_raw.groupby("video_id"):
        if len(group) < 2:
            continue
        group = group.sort_values("created_at")
        like_growth = group["likes"].diff().fillna(0)
        if like_growth.max() > 0:
            spike_idx = like_growth.idxmax()
            spike_time = group.loc[group.index == spike_idx, "created_at"].iloc[0]
            spike_value = like_growth.max()

        results.append({
        "video_id": video_id,
        "spike_time": spike_time,
        "spike_hour": spike_time.floor('h'), # pandas >= 2.0 uses 'h' not 'H'
        "spike_value": spike_value
        })
        spike_df = pd.DataFrame(results)
        spike_counts = spike_df.groupby('spike_hour').size()
        spike_df['cospike_count'] = spike_df['spike_hour'].map(spike_counts)
        spike_df['is_cospike'] = spike_df['cospike_count'] >= 5
        return spike_df

spike_df = detect_cospiking(yt_sample)
print(f"Videos in co-spike windows: {spike_df['is_cospike'].sum()}")

AttributeError: 'str' object has no attribute 'floor'